# Building FAISS Search Indexes from Augmented Memories

This notebook creates a vector search index for each conversation so that benchmark questions can be answered by retrieving the most relevant memories.

**What it does:**
1. Loads the augmented memories JSON (downloaded automatically if missing).
2. Embeds every memory using [EmbeddingGemma-300M](https://huggingface.co/google/embeddinggemma-300m).
3. Builds a [FAISS](https://github.com/facebookresearch/faiss) inner-product index per conversation (L2-normalised vectors, so inner product = cosine similarity).
4. Saves each index as `faiss.index` + `metadata.json` under `indexes_gemma/<conv_id>/`.

**Before you start:** make sure you have installed the dependencies and set `HF_TOKEN` in your `.env` file. See the [README](README.md) for details.

## Setup

Imports, file paths, and model configuration. The cell below loads your `.env` file (for the Hugging Face token) and defines where to read/write data.

You can adjust:
- **`MEMORIES_PATH`** — path to the augmented memories JSON (downloaded automatically if absent).
- **`INDEX_DIR`** — output directory for the FAISS indexes.
- **`MODEL_NAME`** — the Sentence Transformers embedding model to use.
- **`BATCH_SIZE`** — encoding batch size (increase if you have a GPU with spare memory).

In [ ]:
# Import sentence_transformers first to avoid conflict with faiss
from sentence_transformers import SentenceTransformer

In [ ]:
import json
import time
from pathlib import Path
from urllib.request import urlretrieve

import faiss
import numpy as np
from dotenv import load_dotenv
from tqdm.notebook import tqdm

load_dotenv()

# Data (memori-aa-data)
AA_MEMORIES_URL = (
    "https://raw.githubusercontent.com/lborro/memori-aa-data/"
    "refs/heads/main/advanced_augmented_memories.json"
)

MEMORIES_PATH = Path("./advanced_augmented_memories.json")
INDEX_DIR = Path("./indexes_gemma")
INDEX_DIR.mkdir(parents=True, exist_ok=True)


def ensure_download(path: Path, url: str) -> None:
    if path.exists():
        return
    print(f"Downloading {path.name} …")
    urlretrieve(url, path)
    print(f"Saved {path.resolve()}")


# Model & encoding
MODEL_NAME = "google/embeddinggemma-300m"
BATCH_SIZE = 64

# Sanity-check query (section 4)
TOP_K = 5

## 1 · Load augmented memories

Load the augmented memories JSON and print a summary of conversations, sessions, and memory counts. If the file doesn't exist locally, it is downloaded from the repository.

In [ ]:
ensure_download(MEMORIES_PATH, AA_MEMORIES_URL)

with open(MEMORIES_PATH, encoding="utf-8") as f:
    data = json.load(f)

conversations = data["conversations"]
print(f"Loaded {len(conversations)} conversations\n")

for conv in conversations:
    n_memories = sum(len(s["memories"]) for s in conv["sessions"])
    print(
        f"  {conv['conv_id']:>8}: {len(conv['sessions']):>2} sessions | {n_memories:>4} memories"
    )

## 2 · Load the embedding model

Download (first time only) and initialise the Sentence Transformers model used to embed memories. This must be the **same model** used later in the benchmark notebook for retrieval to work correctly.

In [ ]:
model = SentenceTransformer(
    MODEL_NAME,
)
embedding_dim = model.get_sentence_embedding_dimension()
print(f"Embedding model: {MODEL_NAME}")
print(f"Embedding dimension: {embedding_dim}")

## 3 · Build one FAISS index per conversation

For each conversation, the pipeline:

1. **Collects** all memories across every session.
2. **Concatenates** each memory’s `content` and `context` fields into a single text string.
3. **Encodes** the texts into dense vectors with EmbeddingGemma.
4. **Normalises** the vectors (L2) and builds a FAISS `IndexFlatIP` (inner product on unit vectors = cosine similarity).
5. **Saves** `faiss.index` and a `metadata.json` sidecar under `indexes_gemma/<conv_id>/`.

### Helper functions

Three small utilities used by the indexing loop below:
- `memory_text` — merges a memory's `content` and `context` into one string for embedding.
- `build_faiss_index` — L2-normalises embeddings and wraps them in a FAISS inner-product index.
- `save_index` — writes the FAISS index and a JSON metadata sidecar to disk.

In [ ]:
def memory_text(memory: dict) -> str:
    """Combine content + context into a single string for embedding."""
    content = memory.get("content", "")
    context = memory.get("context", "")
    return f"{content} {context}".strip()


def build_faiss_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """Cosine-style similarity: L2-normalise rows, then inner product."""
    normed = embeddings.copy()
    faiss.normalize_L2(normed)
    index = faiss.IndexFlatIP(normed.shape[1])
    index.add(normed)
    return index


def save_index(
    conv_id: str,
    index: faiss.IndexFlatIP,
    documents: list[dict],
    directory: Path,
) -> None:
    """Write FAISS index and metadata JSON for one conversation."""
    conv_dir = directory / conv_id
    conv_dir.mkdir(parents=True, exist_ok=True)
    faiss.write_index(index, str(conv_dir / "faiss.index"))
    with open(conv_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump({"conv_id": conv_id, "documents": documents}, f)

### Run indexing

Loop over all conversations, embed their memories, build a FAISS index for each, and save to disk. Progress is shown with a progress bar. On a CPU this typically takes 15-30 seconds for 10 conversations.

In [ ]:
t0 = time.perf_counter()

for conv in tqdm(conversations, desc="Building indexes"):
    conv_id = conv["conv_id"]
    documents: list[dict] = []
    texts: list[str] = []

    for session in conv["sessions"]:
        session_id = session["session_id"]
        summary = session.get("summary", "")
        timestamp = session.get("timestamp", "")

        for mem in session["memories"]:
            texts.append(memory_text(mem))
            documents.append(
                {
                    **mem,
                    "_session": session_id,
                    "_summary": summary,
                    "_timestamp": timestamp,
                }
            )

    if not texts:
        print(f"  {conv_id}: no memories – skipped")
        continue

    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        normalize_embeddings=False,
    )
    embeddings = np.asarray(embeddings, dtype=np.float32)

    index = build_faiss_index(embeddings)
    save_index(conv_id, index, documents, INDEX_DIR)
    print(
        f"  {conv_id}: {len(documents):>4} memories indexed  (dim={embeddings.shape[1]})"
    )

elapsed = time.perf_counter() - t0
print(f"\nDone – {len(conversations)} indexes built in {elapsed:.1f}s")
print(f"Saved to: {INDEX_DIR.resolve()}")

## 4 · Sanity check

Quick smoke test: reload one of the indexes we just built and run a sample query against it. If the top results are semantically relevant to the query, the index was built correctly and is ready for the benchmark notebook.

If the results look wrong or empty, check that `INDEX_DIR` points to the right folder and that the embedding model loaded without errors.

In [ ]:
test_conv_id = conversations[0]["conv_id"]
test_dir = INDEX_DIR / test_conv_id

loaded_index = faiss.read_index(str(test_dir / "faiss.index"))
with open(test_dir / "metadata.json", encoding="utf-8") as f:
    meta = json.load(f)

print(f"Index for {test_conv_id}: {loaded_index.ntotal} vectors, dim={loaded_index.d}")

query = "Caroline was helped by mental health support?"
q_emb = model.encode([query], normalize_embeddings=True).astype(np.float32)

distances, indices = loaded_index.search(q_emb, TOP_K)

print(f"\nQuery: {query}\n")
for rank, (dist, idx) in enumerate(zip(distances[0], indices[0], strict=True), start=1):
    doc = meta["documents"][idx]
    print(f"  [{rank}] score={dist:.4f}")
    print(f"      content: {doc['content']}")
    print(f"      context: {doc['context']}\n")